# 2장 2강: 시계열 데이터 모델링과 예측

## 실습 목표

이번 실습에서는 **Bike Sharing Demand** 데이터를 이용하여 시계열 예측의 기본 흐름을 연습합니다.

- 시간 순서를 유지하면서 **Train / Test 데이터**를 분리합니다.
- **AR, I, MA**가 무엇을 의미하는지 확인합니다.
- **ARIMA(2, 1, 1)** 모델을 학습합니다.
- Test 기간의 미래 값을 예측합니다.
- 실제값과 예측값을 비교합니다.
- **MAE, RMSE**를 이용해 예측 성능을 평가합니다.

> 이 실습에서는 2장 2강 교안에 나온 내용만 사용합니다.

## 실습 환경 / 데이터

- Python
- pandas
- matplotlib
- statsmodels
- scikit-learn
- 데이터: `data/Bike_Sharing_Demand.csv`

### 주요 컬럼

| 컬럼 | 의미 |
|---|---|
| `datetime` | 자전거 대여 기록 시각 |
| `casual` | 비등록 사용자 대여 수 |
| `registered` | 등록 사용자 대여 수 |
| `count` | 전체 자전거 대여 수 |

이번 강에서는 월별 대여량 `count`를 시계열로 사용합니다.

## 실습 준비

다음 순서로 데이터를 준비합니다.

1. 데이터 불러오기
2. `datetime` 자료형 확인
3. 결측치 확인
4. `datetime`을 날짜/시간 자료형으로 변환
5. 시간 순서대로 정렬
6. 월별 대여량 `count` 준비

> 월별 집계 코드는 시계열 모델링을 위한 준비 코드로 제공됩니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

bike = pd.read_csv("data/Bike_Sharing_Demand.csv")

print("데이터 크기:", bike.shape)
display(bike.head())

print("\n[자료형]")
display(bike[["datetime", "count"]].dtypes.to_frame("dtype"))

print("\n[결측치]")
display(bike[["datetime", "count"]].isna().sum().to_frame("missing"))

In [ ]:
bike["datetime"] = pd.to_datetime(bike["datetime"])
bike = bike.sort_values("datetime").reset_index(drop=True)

bike["month"] = (
    bike["datetime"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

monthly = (
    bike.groupby("month", as_index=False)["count"]
    .sum()
)

monthly = monthly.set_index("month")

print("월별 데이터 개수:", len(monthly))
display(monthly.head())
display(monthly.tail())

---

# 필수 1. 시간 순서를 유지한 Train / Test 분리

## 배경

시계열 데이터에서는 데이터를 무작위로 섞으면 미래 시점의 데이터가 Train에 포함될 수 있습니다.

따라서 **과거 데이터를 Train**, **그 이후 데이터를 Test**로 사용해야 합니다.

이번 실습에서는 월별 대여량의 **마지막 6개월을 Test 데이터**로 사용합니다.

## 문제 1. Train / Test 데이터를 분리하세요.

### 요구사항

1. `monthly`의 마지막 6개 행을 Test 데이터로 사용합니다.
2. 나머지 데이터를 Train 데이터로 사용합니다.
3. Train과 Test의 시작일과 종료일을 출력합니다.
4. Train과 Test를 같은 그래프에 표시합니다.

### 확인 포인트

- 시간 순서를 유지했는가?
- Test 데이터가 Train 데이터보다 뒤의 기간인가?
- 무작위 분리를 사용하지 않았는가?

In [ ]:
# TODO
# monthly 데이터를 Train / Test로 분리하세요.
train = monthly.iloc[:-6]
test = monthly.iloc[-6:]

print("Train:", train.index.min(), "~", train.index.max(), "n=", len(train))
print("Test:", test.index.min(), "~", test.index.max(), "n=", len(test))

plt.figure(figsize=(10, 4))
plt.plot(train.index, train["count"], label="Train")
plt.plot(test.index, test["count"], label="Test")
plt.title("Monthly count Train / Test")
plt.legend()
plt.tight_layout()
plt.show()


### Q1. 시계열 데이터에서 무작위로 Train / Test를 분리하면 안 되는 이유는 무엇인가요?

**답변:**  
미래 정보가 과거에 섞여 들어가 데이터 누수가 생기고, 실제 예측 상황(과거→미래)과 다르게 평가되기 때문입니다.


---

# 필수 2. ARIMA 모델 학습과 미래 값 예측

## 배경

ARIMA는 다음 세 가지 요소로 구성됩니다.

- **AR(p)**: 과거의 실제 값을 몇 개 사용할 것인가?
- **I(d)**: 차분을 몇 번 적용할 것인가?
- **MA(q)**: 과거의 예측 오차를 몇 개 사용할 것인가?

이번 실습에서는 교안과 동일하게 `ARIMA(2, 1, 1)`을 사용합니다.

## 문제 1. ARIMA(2, 1, 1)의 의미를 설명하세요.

### 요구사항

다음 세 값을 각각 설명합니다.

- `p = 2`
- `d = 1`
- `q = 1`

### Q2. 각각 무엇을 의미하나요?

**답변:**  
- p=2: 과거 실제값 2개를 AR에 사용
- d=1: 1차 차분
- q=1: 과거 예측 오차 1개를 MA에 사용


## 문제 2. Train 데이터로 ARIMA 모델을 학습하세요.

### 요구사항

1. `statsmodels.tsa.arima.model`에서 `ARIMA`를 불러옵니다.
2. Train 데이터의 `count`를 사용합니다.
3. `order=(2, 1, 1)`을 지정합니다.
4. `fit()`으로 모델을 학습합니다.

### 확인 포인트

- Test 데이터를 학습에 사용하지 않았는가?
- `order=(p, d, q)`의 순서를 알고 있는가?

In [ ]:
# TODO
# ARIMA(2, 1, 1) 모델을 생성하고 학습하세요.
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(train["count"], order=(2, 1, 1))
model_fit = model.fit()
print(model_fit.summary())


## 문제 3. Test 기간만큼 미래 값을 예측하세요.

### 요구사항

1. `forecast()`를 사용합니다.
2. `steps=len(test)`로 Test 기간과 같은 개수만큼 예측합니다.
3. 예측값의 인덱스를 Test 데이터의 인덱스와 동일하게 설정합니다.
4. `actual`, `predicted` 컬럼을 가진 DataFrame을 만듭니다.
5. 실제값과 예측값을 같은 그래프에 표시합니다.

### 확인 포인트

- Train 마지막 시점 이후의 값을 예측했는가?
- Test 길이와 예측값 개수가 같은가?
- Actual과 Predicted를 구분했는가?

In [ ]:
# TODO
# Test 기간만큼 예측하고 Actual / Predicted를 비교하세요.
forecast_values = model_fit.forecast(steps=len(test))
forecast_values.index = test.index

result = pd.DataFrame({
    "actual": test["count"],
    "predicted": forecast_values
})
display(result)

plt.figure(figsize=(10, 4))
plt.plot(result.index, result["actual"], label="Actual")
plt.plot(result.index, result["predicted"], label="Predicted")
plt.title("ARIMA(2,1,1) Actual vs Predicted")
plt.legend()
plt.tight_layout()
plt.show()


### Q3. `steps=len(test)`는 무엇을 의미하나요?

**답변:**  
Test 구간의 길이만큼(여기선 6개월) 미래 값을 예측하라는 뜻입니다.


### Q4. Actual과 Predicted는 각각 무엇을 의미하나요?

**답변:**  
Actual은 Test 구간의 실제 대여량, Predicted는 모델이 예측한 대여량입니다.


---

# 과제 1. 정상성 진단을 반영한 ARIMA 예측과 예측 신뢰도 평가

## 배경

필수 1의 Train/Test 분리는 유지합니다. 과제에서는 Train 데이터의 정상성을 먼저 진단한 뒤 ARIMA의 차분 차수 d를 선택하여 모델을 다시 학습합니다. MAE·RMSE와 함께 **95% 예측 구간의 하한·상한**을 실제값과 비교합니다.

## 요구사항

1. 필수 1에서 만든 `train`, `test`를 사용합니다. 정상성 진단과 모델 설정에는 Train만 사용합니다.
2. Train의 `count`와 최근 3개 관측값의 이동평균을 그래프로 확인합니다. 원본에 ADF Test를 수행하고 검정통계량과 p-value를 출력합니다. 짧은 월별 자료이므로 이번 과제의 검정 설정은 `maxlag=1, regression="c", autolag="AIC"`로 통일합니다.
3. 유의수준 0.05에서 원본의 단위근 귀무가설을 기각하면 d=0을 우선 검토합니다. 기각하지 못하면 1차 차분한 Train에 같은 설정으로 ADF Test를 다시 수행하고 차분 그래프를 확인합니다. 차분 후 기각하면 d=1을 선택합니다. 차분 후에도 기각하지 못하면 추가 진단이 필요함을 명시하고, 이번 과제에서는 d=1을 잠정 설정하여 예측 과정과 한계를 함께 평가합니다. 선택 근거에 그래프 해석도 포함합니다.
4. **차분하지 않은 Train의 원본 `count`**를 `ARIMA(order=(2, d, 1))`에 전달해 학습합니다. 차분은 모델 내부에서 처리하므로 차분 데이터에 d=1을 다시 적용하지 않습니다. 필수 2의 모델·결과와 구분하여 과제용 변수를 사용합니다.
5. `get_forecast(steps=len(test))`로 Test 기간의 예측값과 95% 예측 구간을 구합니다. `predicted_mean`과 `conf_int(alpha=0.05)`를 활용합니다.
6. Test 날짜에 맞춰 `actual`, `predicted`, `lower_95`, `upper_95`를 가진 `task_result`를 만들고 출력합니다.
7. 과제 모델의 MAE·RMSE를 계산하고 의미를 설명합니다.
8. 실제값·예측값·95% 예측 구간을 같은 그래프에 표시합니다. 시점별 구간 폭과 실제값의 구간 포함 여부를 표에 추가하고, 구간 밖인 시점을 확인합니다.
9. 실제값과 구간의 비교 및 구간 폭을 근거로 예측 신뢰도를 해석합니다. 신뢰도가 낮다고 판단한 구간의 가능한 원인 또는 모델의 한계를 한 가지 이상 설명합니다. 구간 밖인 시점이 없더라도 구간 폭과 자료의 한계를 검토합니다.

> Test는 최종 평가에만 사용합니다. Test 결과를 보고 d나 다른 모델 설정을 다시 선택하지 않습니다. 월별 값은 제공된 CSV의 관측값 합계이며, 월 전체 날짜가 없으면 완전한 월간 총수요로 해석하지 않습니다.

### 확인 포인트

- Train만으로 정상성을 진단하고 d 선택 근거를 제시했는가?
- 1차 차분 후에도 귀무가설을 기각하지 못한 경우 잠정 모델의 한계를 밝혔는가?
- 원본 Train을 모델에 입력하여 이중 차분을 피했는가?
- MAE·RMSE와 실제값·예측값·예측 구간을 함께 확인했는가?
- 구간 밖 실제값과 구간 폭을 근거로 예측 신뢰도와 한계를 설명했는가?
- Q5~Q9에 모두 답했는가?


In [ ]:
# TODO
# Train 정상성 진단 → d 선택 → 과제용 ARIMA 학습
# → 점예측·95% 예측 구간 → MAE·RMSE → 신뢰도 및 한계 해석
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

train_ma3 = train["count"].rolling(window=3).mean()
plt.figure(figsize=(10, 4))
plt.plot(train.index, train["count"], label="Train count")
plt.plot(train.index, train_ma3, label="MA(3)")
plt.legend()
plt.title("Train count and rolling mean(3)")
plt.tight_layout()
plt.show()

adf_train = adfuller(train["count"], maxlag=1, regression="c", autolag="AIC")
print("ADF(원본):", adf_train[0], "p-value:", adf_train[1])

d = 0
if adf_train[1] >= 0.05:
    train_diff = train["count"].diff().dropna()
    plt.figure(figsize=(10, 4))
    plt.plot(train_diff.index, train_diff.values)
    plt.title("Train 1st difference")
    plt.tight_layout()
    plt.show()
    adf_diff = adfuller(train_diff, maxlag=1, regression="c", autolag="AIC")
    print("ADF(1차 차분):", adf_diff[0], "p-value:", adf_diff[1])
    if adf_diff[1] < 0.05:
        d = 1
    else:
        print("차분 후에도 기각 실패 → 과제에서는 d=1을 잠정 설정합니다.")
        d = 1
else:
    d = 0

print("선택한 d:", d)

task_model = ARIMA(train["count"], order=(2, d, 1))
task_fit = task_model.fit()

fc = task_fit.get_forecast(steps=len(test))
pred_mean = fc.predicted_mean
conf = fc.conf_int(alpha=0.05)
# conf columns may vary
lower = conf.iloc[:, 0]
upper = conf.iloc[:, 1]

task_result = pd.DataFrame({
    "actual": test["count"].values,
    "predicted": pred_mean.values,
    "lower_95": lower.values,
    "upper_95": upper.values,
}, index=test.index)
task_result["width"] = task_result["upper_95"] - task_result["lower_95"]
task_result["in_interval"] = (
    (task_result["actual"] >= task_result["lower_95"])
    & (task_result["actual"] <= task_result["upper_95"])
)
display(task_result)

mae_task = mean_absolute_error(task_result["actual"], task_result["predicted"])
rmse_task = np.sqrt(mean_squared_error(task_result["actual"], task_result["predicted"]))
print("MAE:", mae_task)
print("RMSE:", rmse_task)
print("구간 밖 시점:")
display(task_result.loc[~task_result["in_interval"]])

plt.figure(figsize=(10, 4))
plt.plot(task_result.index, task_result["actual"], label="actual")
plt.plot(task_result.index, task_result["predicted"], label="predicted")
plt.fill_between(task_result.index, task_result["lower_95"], task_result["upper_95"], alpha=0.2, label="95% PI")
plt.legend()
plt.title("Task ARIMA forecast with 95% PI")
plt.tight_layout()
plt.show()


### Q5. MAE는 어떻게 해석할 수 있나요?

**답변:**
예측값과 실제값 차이의 절댓값 평균으로, 전형적인 오차 크기(대여량 단위)를 나타냅니다. 위 셀의 MAE 출력값을 기준으로 해석합니다.


### Q6. RMSE가 MAE보다 큰 오차에 더 민감한 이유는 무엇인가요?

**답변:**
제곱을 사용하므로 큰 오차가 평균에 더 크게 반영되기 때문입니다.


### Q7. Train 데이터의 ADF 검정 결과와 그래프를 근거로 d를 어떻게 선택했나요? 정상성 판단에 남아 있는 한계는 무엇인가요?

**답변:**
Train 원본 ADF p-value와 이동평균 그래프를 보고 비정상이면 1차 차분 ADF를 확인한 뒤 d를 선택했습니다(코드의 `d` 출력 참고). 월별 표본이 짧아 ADF 검정력·계절성 반영에 한계가 있습니다.


### Q8. Test 실제값이 95% 예측 구간 밖에 나타난 시점은 언제이며, 구간 폭은 시점별로 어떻게 달라지나요? 이를 근거로 예측 신뢰도를 설명하세요.

**답변:**
`task_result`에서 `in_interval==False`인 시점과 `width` 변화를 근거로 해석합니다. 미래로 갈수록 구간이 넓어지면 불확실성이 커진다는 신호입니다.


### Q9. 예측 신뢰도가 낮다고 판단한 시점 또는 구간은 어디인가요? 가능한 원인이나 모델의 한계를 한 가지 이상 설명하세요.

**답변:**
구간 폭이 크거나 실제값이 구간 밖인 시점을 낮은 신뢰도 구간으로 봅니다. 원인 예: 짧은 월별 시계열, 단순 ARIMA가 포착하지 못한 외부 요인/계절성, 분산 증가 등.


---

# 실습 마무리

1. 시계열 Train/Test 분리에서 가장 중요한 원칙은 무엇인가요? → 시간 순서를 유지하고 미래가 과거에 섞이지 않게 합니다.
2. ARIMA의 `p`, `d`, `q`는 각각 무엇을 의미하나요? → AR 차수, 차분 횟수, MA 차수입니다.
3. `forecast(steps=6)`은 무엇을 의미하나요? → 앞으로 6시점 예측입니다.
4. Actual과 Predicted는 각각 무엇인가요? → 실제값과 모델 예측값입니다.
